# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame

np.set_printoptions(precision=3, suppress=True)


## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [2]:
ptr_local_positions = [
    # chosen tetrahedron 
    vct3(20, 20, 20),
    vct3(20, -20, -20),
    vct3(-20,  20, -20),
    vct3(-20, -20, 20)
    
]
ptr_tip_approx = vct3(0, 0, -100)  # the tip's approximate position in the pointer's local frame


## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [3]:
cal_local_positions = [
    # same dimension tetrahedron as pointer's marker body
    vct3(20, 20, 20),
    vct3(20, -20, -20),
    vct3(-20,  20, -20),
    vct3(-20, -20, 20)
]



## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [4]:
F_tracker_approx = Frame(Rot(), vct3(0.0, 0.0, 0.0)) # Frame, tracker pose relative to the workspace
F_cal_approx = Frame(Rot.xyz(0.0, 0.0, 0.0), vct3(300.0, 300.0, 1400.0)) # Frame, calibration object pose relative to the workspace

## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [5]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions)
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions)

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "workspace placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb)
trk = simulation.create_tracker("trk1", F_tracker_approx)

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [6]:
def refine_marker_body(trk, placed, n_readings=30, passes=3):
    """
    Improve the estimated locations of marker balls on one rigid body.
    
    The tracker measure seach marker ball many times.
    Each measurement is then converted from tracker coordinates back into body coordinates.
    We average these estimates and update the body's marker location. 

    Parameters
    trk : Tracker
        tracker to collect observations.
    placed : Placed Body 
        pointer or calibration object whose marker coordinates are updated
    n_readings : int, default=30
        number of independent tracker samples to average during each pass
    passes : int, default=3
        how many times to repeat the refinement process.

    Returns
    None
        Updates placed.body.nominal_marker_positions in place.
    """

    if n_readings < 1:
        raise ValueError("numberOfReadings must be at least 1.")

    if passes < 1:
        raise ValueError("numberOfPasses must be at least 1.")

    markerCount = len(placed.body.nominal_marker_positions)

    if markerCount < 4:
        raise ValueError("The marker body must contain at least four markers.")

    for passNumber in range(passes):
        estimatesForEachMarker = []

        for markerNumber in range(markerCount):
            estimatesForEachMarker.append([])

        for readingNumber in range(n_readings):
            sample = simulation.sample_marker_bodies(trk, [placed])[placed.body.name]

            # converts tracker-coordinate points back into body coordinates
            trackerToBody = sample.frame.inv()

            for markerNumber in range(markerCount):
                observedMarker = sample.markers[markerNumber]
                markerInBodyCoordinates = trackerToBody * observedMarker

                estimatesForEachMarker[markerNumber].append(
                    markerInBodyCoordinates.p.vec.ravel()
                )

        refinedPositions = []

        for markerNumber in range(markerCount):
            allEstimatesForThisMarker = estimatesForEachMarker[markerNumber]
            averagePosition = np.mean(allEstimatesForThisMarker, axis=0)
            refinedPositions.append(vct3(averagePosition))

        placed.body.set_marker_positions(refinedPositions)

refine_marker_body(trk, ptr_placed)
refine_marker_body(trk, cal_placed)


TypeError: unsupported operand type(s) for *: 'Frame' and 'uvct3'

## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [ ]:
# Each orientation gives a different geometric constraint on the unknown tip.
desiredOrientations = [
    Rot.xyz(0, 0, 0),

    Rot.xyz(np.pi / 6, 0, 0),
    Rot.xyz(-np.pi / 6, 0, 0),
    Rot.xyz(np.pi / 3, 0, 0),
    Rot.xyz(-np.pi / 3, 0, 0),

    Rot.xyz(0, np.pi / 6, 0),
    Rot.xyz(0, -np.pi / 6, 0),
    Rot.xyz(0, np.pi / 3, 0),
    Rot.xyz(0, -np.pi / 3, 0),

    Rot.xyz(0, 0, np.pi / 6),
    Rot.xyz(0, 0, -np.pi / 6),
    Rot.xyz(0, 0, np.pi / 3),
    Rot.xyz(0, 0, -np.pi / 3),

    Rot.xyz(np.pi / 6, np.pi / 6, 0),
    Rot.xyz(np.pi / 6, -np.pi / 6, 0),
    Rot.xyz(-np.pi / 6, np.pi / 6, 0),
    Rot.xyz(-np.pi / 6, -np.pi / 6, 0),

    Rot.xyz(np.pi / 6, 0, np.pi / 6),
    Rot.xyz(-np.pi / 6, 0, -np.pi / 6),

    Rot.xyz(0, np.pi / 6, np.pi / 6),
    Rot.xyz(0, -np.pi / 6, -np.pi / 6),
]

numberOfObservations = len(desiredOrientations)

F_ptr_list = []
F_cal_list = []
sensed_tip_list = []

for observationNumber in range(numberOfObservations):
    desiredOrientation = desiredOrientations[observationNumber]

    handGuideResult = simulation.hand_guide_pointer(
        ptr,
        ptr_placed,
        cal_placed,
        desiredOrientation
    )

    # sample both bodies at the same instant
    trackerObservation = simulation.sample_marker_bodies(
        trk,
        [ptr_placed, cal_placed]
    )

    sensedTip = simulation.sense_tip(ptr, handGuideResult)

    F_ptr_list.append(trackerObservation["ptr_mb"].frame)
    F_cal_list.append(trackerObservation["cal_mb"].frame)
    sensed_tip_list.append(sensedTip)

numberOfSensorReadings = 0

for sensedTip in sensed_tip_list:
    if sensedTip is not None:
        numberOfSensorReadings += 1

print("Number of calibration observations:", len(F_ptr_list))
print("Number of sensor readings obtained:", numberOfSensorReadings)

Number of calibration observations: 21
Number of sensor readings obtained: 21


`sensed_tip_list` holds each observation's high-accuracy calibration-sensor
reading (or `None` if that observation ended up outside the sensor's 5mm
range) — useful as an independent sanity check on your calibration later,
since it measures the tip position a completely different way (directly,
rather than through the pointer's marker geometry).


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

**Corner case to consider**: What are the corner cases you want to analyse here?

**Minimum data**: how many observations, and what about their rotations,
are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [ ]:
def compute_pivot_calibration(F_ptr_list, F_cal_list):
    """
    Estimate the pointer tip position using paired pointer and calibration
    object poses from multiple pivot-calibration observations.

    Parameters
    ----------
    F_ptr_list : list of Frame
        Pointer marker-body poses in tracker coordinates.
    F_cal_list : list of Frame
        Calibration-object poses in tracker coordinates. Each entry must
        correspond to the same-index entry in F_ptr_list.

    Returns
    -------
    p_tip : vct3
        Estimated pointer-tip position in pointer marker-body coordinates.
    cov_estimate : numpy.ndarray
        Estimated 3 x 3 covariance matrix for p_tip.
    """
    numberOfObservations = len(F_ptr_list)

    if numberOfObservations != len(F_cal_list):
        raise ValueError(
            "The pointer and calibration frame lists must have equal length."
        )

    if numberOfObservations < 3:
        raise ValueError(
            "At least three varied observations are required."
        )

    systemMatrix = np.zeros((3 * numberOfObservations, 6))
    rightSide = np.zeros(3 * numberOfObservations)

    for observationNumber in range(numberOfObservations):
        pointerFrame = F_ptr_list[observationNumber]
        calibrationFrame = F_cal_list[observationNumber]

        # Pointer pose expressed relative to the calibration object.
        pointerRelativeToCalibration = (
            calibrationFrame.inv() * pointerFrame
        )

        rotationMatrix = pointerRelativeToCalibration.R.matrix
        translationVector = pointerRelativeToCalibration.p.vec.ravel()

        firstRow = 3 * observationNumber
        lastRow = firstRow + 3

        # Rj * p_tip - pivotPoint = -tj
        systemMatrix[firstRow:lastRow, 0:3] = rotationMatrix
        systemMatrix[firstRow:lastRow, 3:6] = -np.eye(3)
        rightSide[firstRow:lastRow] = -translationVector

    leastSquaresResult = np.linalg.lstsq(
        systemMatrix,
        rightSide,
        rcond=None
    )

    solution = leastSquaresResult[0]
    matrixRank = leastSquaresResult[2]
    singularValues = leastSquaresResult[3]

    if matrixRank < 6:
        raise ValueError(
            "The pointer orientations are not sufficiently varied "
            "for pivot calibration."
        )

    maximumConditionNumber = 1e8
    conditionNumber = singularValues[0] / singularValues[-1]

    if conditionNumber > maximumConditionNumber:
        raise ValueError(
            "The orientation data are poorly conditioned. "
            "Collect more varied pointer orientations."
        )

    tipCoordinates = solution[0:3]
    p_tip = vct3(tipCoordinates)

    residualVector = systemMatrix @ solution - rightSide
    degreesOfFreedom = 3 * numberOfObservations - 6
    residualVariance = np.sum(residualVector ** 2) / degreesOfFreedom

    parameterCovariance = residualVariance * np.linalg.pinv(
        systemMatrix.T @ systemMatrix
    )

    cov_estimate = parameterCovariance[0:3, 0:3]

    return p_tip, cov_estimate


p_tip_est, p_tip_cov = compute_pivot_calibration(
    F_ptr_list,
    F_cal_list
)

print("Calibrated pointer-tip position:", p_tip_est)
print("Estimated covariance matrix:")
print(p_tip_cov)

Calibrated pointer-tip position: vct3(-0.15765, -0.967992, -102.757)
Estimated covariance matrix:
[[0.372 0.023 0.022]
 [0.023 0.382 0.027]
 [0.022 0.027 0.352]]


In [ ]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))


calibrated p_tip: vct3(-0.15765, -0.967992, -102.757)
estimated covariance (mm^2):
 [[0.372 0.023 0.022]
 [0.023 0.382 0.027]
 [0.022 0.027 0.352]]
estimated std-dev per axis (mm): [0.61  0.618 0.593]


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [ ]:
# A stationary reference body placed just above the test-pose cube.
near_local_positions = [
    vct3(20, 20, 20),
    vct3(20, -20, -20),
    vct3(-20, 20, -20),
    vct3(-20, -20, 20),
]

F_near_approx = Frame(
    Rot.xyz(0, 0, 0),
    vct3(300, 300, 550)
)

near_mb = simulation.define_marker_body(
    "near_mb",
    near_local_positions
)

near_placed = simulation.place_marker_body(
    near_mb,
    F_near_approx
)

refine_marker_body(trk, near_placed)

Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [ ]:
thetas = [
    0,
    np.pi / 6,
    -np.pi / 6,
    np.pi / 3,
    -np.pi / 3,
    np.pi / 2,
    -np.pi / 2,
]

offsets = [0, 100, -100, 200, -200]

zeroAngle = thetas[0]
positiveThirtyDegrees = thetas[1]
negativeThirtyDegrees = thetas[2]

centerOffset = offsets[0]
positiveOneHundred = offsets[1]
negativeOneHundred = offsets[2]
positiveTwoHundred = offsets[3]
negativeTwoHundred = offsets[4]

# Representative subset of the supplied orientation and position grid.
test_poses = [
    Frame(
        Rot.xyz(zeroAngle, zeroAngle, zeroAngle),
        vct3(300 + centerOffset, 300 + centerOffset, 300 + centerOffset)
    ),

    Frame(
        Rot.xyz(positiveThirtyDegrees, zeroAngle, zeroAngle),
        vct3(
            300 + negativeTwoHundred,
            300 + negativeTwoHundred,
            300 + negativeTwoHundred
        )
    ),

    Frame(
        Rot.xyz(negativeThirtyDegrees, zeroAngle, zeroAngle),
        vct3(
            300 + positiveTwoHundred,
            300 + negativeTwoHundred,
            300 + negativeTwoHundred
        )
    ),

    Frame(
        Rot.xyz(zeroAngle, positiveThirtyDegrees, zeroAngle),
        vct3(
            300 + negativeTwoHundred,
            300 + positiveTwoHundred,
            300 + negativeTwoHundred
        )
    ),

    Frame(
        Rot.xyz(zeroAngle, negativeThirtyDegrees, zeroAngle),
        vct3(
            300 + positiveTwoHundred,
            300 + positiveTwoHundred,
            300 + negativeTwoHundred
        )
    ),

    Frame(
        Rot.xyz(zeroAngle, zeroAngle, positiveThirtyDegrees),
        vct3(
            300 + negativeTwoHundred,
            300 + negativeTwoHundred,
            300 + positiveTwoHundred
        )
    ),

    Frame(
        Rot.xyz(zeroAngle, zeroAngle, negativeThirtyDegrees),
        vct3(
            300 + positiveTwoHundred,
            300 + negativeTwoHundred,
            300 + positiveTwoHundred
        )
    ),

    Frame(
        Rot.xyz(
            positiveThirtyDegrees,
            positiveThirtyDegrees,
            zeroAngle
        ),
        vct3(
            300 + negativeTwoHundred,
            300 + positiveTwoHundred,
            300 + positiveTwoHundred
        )
    ),

    Frame(
        Rot.xyz(
            negativeThirtyDegrees,
            negativeThirtyDegrees,
            zeroAngle
        ),
        vct3(
            300 + positiveTwoHundred,
            300 + positiveTwoHundred,
            300 + positiveTwoHundred
        )
    ),

    Frame(
        Rot.xyz(positiveThirtyDegrees, zeroAngle, zeroAngle),
        vct3(300 + negativeTwoHundred, 300 + centerOffset, 300 + centerOffset)
    ),

    Frame(
        Rot.xyz(negativeThirtyDegrees, zeroAngle, zeroAngle),
        vct3(300 + positiveTwoHundred, 300 + centerOffset, 300 + centerOffset)
    ),

    Frame(
        Rot.xyz(zeroAngle, positiveThirtyDegrees, zeroAngle),
        vct3(300 + centerOffset, 300 + negativeTwoHundred, 300 + centerOffset)
    ),

    Frame(
        Rot.xyz(zeroAngle, negativeThirtyDegrees, zeroAngle),
        vct3(300 + centerOffset, 300 + positiveTwoHundred, 300 + centerOffset)
    ),

    Frame(
        Rot.xyz(zeroAngle, zeroAngle, positiveThirtyDegrees),
        vct3(300 + centerOffset, 300 + centerOffset, 300 + negativeTwoHundred)
    ),

    Frame(
        Rot.xyz(zeroAngle, zeroAngle, negativeThirtyDegrees),
        vct3(300 + centerOffset, 300 + centerOffset, 300 + positiveTwoHundred)
    ),
]

test_results = []

for F_t_nominal in test_poses:
    test_placed = simulation.place_marker_body(
        ptr_mb,
        F_t_nominal,
        precision="precise"
    )

    # One simultaneous tracker observation, as required by the handout.
    obs = simulation.sample_marker_bodies(
        trk,
        [test_placed, near_placed]
    )

    F_ptr_observed = obs["ptr_mb"].frame
    F_near_observed = obs["near_mb"].frame

    F_ptr_relative_to_near = (
        F_near_observed.inv() * F_ptr_observed
    )

    predicted_tip_relative_to_near = (
        F_ptr_relative_to_near * p_tip_est
    )

    actualTipInWorkspace = (
        test_placed.actual_placement * ptr.get_actual_tip_position().p
    )

    # Ground truth for reporting only: map the refined reference-marker
    # coordinate system to its hidden actual marker locations in the workspace.
    actualNearMarkersInWorkspace = []

    for actualNearMarker in near_mb.get_actual_marker_positions():
        markerInWorkspace = (
            near_placed.actual_placement * actualNearMarker
        )
        actualNearMarkersInWorkspace.append(markerInWorkspace)

    trueNearFrame = trk.procrustes_solver(
        near_mb.nominal_marker_positions,
        actualNearMarkersInWorkspace
    )

    actualTipRelativeToNear = (
        trueNearFrame.inv() * actualTipInWorkspace
    )
    error_mm = (
    predicted_tip_relative_to_near
    - actualTipRelativeToNear
).norm()

    test_results.append((F_t_nominal, error_mm))

print("Number of Step 8 test poses:", len(test_results))

Number of Step 8 test poses: 15


## Achieved performance

Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [ ]:
requiredAccuracyMm = 2.0

if len(test_results) == 0:
    raise ValueError("No Step 8 test results are available.")

errorValues = []

for testPose, errorMm in test_results:
    errorValues.append(errorMm)

numberOfTests = len(errorValues)
meanErrorMm = np.mean(errorValues)
standardDeviationMm = np.std(errorValues, ddof=1)
minimumErrorMm = np.min(errorValues)
maximumErrorMm = np.max(errorValues)

numberMeetingRequirement = 0

for errorMm in errorValues:
    if errorMm <= requiredAccuracyMm:
        numberMeetingRequirement += 1

percentageMeetingRequirement = (
    100 * numberMeetingRequirement / numberOfTests
)

print("Step 8 individual test-pose errors")
print("----------------------------------")

for testNumber in range(numberOfTests):
    testPose = test_results[testNumber][0]
    errorMm = test_results[testNumber][1]

    position = testPose.p.vec.ravel()

    print(
        "Test",
        testNumber + 1,
        "| position =",
        position,
        "| error =",
        round(errorMm, 3),
        "mm"
    )

print()
print("Achieved performance summary")
print("----------------------------")
print("Number of test poses:", numberOfTests)
print("Mean error:", round(meanErrorMm, 3), "mm")
print("Standard deviation:", round(standardDeviationMm, 3), "mm")
print("Minimum error:", round(minimumErrorMm, 3), "mm")
print("Maximum error:", round(maximumErrorMm, 3), "mm")
print(
    "Poses meeting the 2 mm requirement:",
    numberMeetingRequirement,
    "of",
    numberOfTests
)
print(
    "Percentage meeting the requirement:",
    round(percentageMeetingRequirement, 1),
    "%"
)

if numberMeetingRequirement == numberOfTests:
    print("Result: all tested poses meet the 2 mm requirement.")
else:
    print("Result: not all tested poses meet the 2 mm requirement.")

Step 8 individual test-pose errors
----------------------------------
Test 1 | position = [300. 300. 300.] | error = 0.976 mm
Test 2 | position = [100. 100. 100.] | error = 0.999 mm
Test 3 | position = [500. 100. 100.] | error = 0.935 mm
Test 4 | position = [100. 500. 100.] | error = 1.006 mm
Test 5 | position = [500. 500. 100.] | error = 0.918 mm
Test 6 | position = [100. 100. 500.] | error = 0.877 mm
Test 7 | position = [500. 100. 500.] | error = 0.939 mm
Test 8 | position = [100. 500. 500.] | error = 1.034 mm
Test 9 | position = [500. 500. 500.] | error = 0.891 mm
Test 10 | position = [100. 300. 300.] | error = 0.756 mm
Test 11 | position = [500. 300. 300.] | error = 0.962 mm
Test 12 | position = [300. 100. 300.] | error = 0.977 mm
Test 13 | position = [300. 500. 300.] | error = 0.898 mm
Test 14 | position = [300. 300. 100.] | error = 0.873 mm
Test 15 | position = [300. 300. 500.] | error = 0.903 mm

Achieved performance summary
----------------------------
Number of test poses: 15
